# Notebook 01 - Data Extraction And Validation

This notebook prepares the United States real GDP series from 1920 onward and validates overlapping annual growth rates against FRED/BEA when the validation file is available.

In [1]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
PROJECT_ROOT = next(
    path for path in _candidates
    if (path / "pyproject.toml").exists() and (path / "src" / "us_gdp_regime").exists()
)
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

## Research Question

When did United States real GDP grow above or below its own long-run average, and are the source data stable enough to support regime analysis?

## Data-Source Rationale

Maddison Project Database 2023 is the default historical source because it provides long-run GDP per capita and population estimates that cover 1920 onward. FRED/BEA `GDPCA` starts in 1929 and is therefore used as a validation source for modern national accounts rather than as the default 1920-onward series.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from us_gdp_regime.config import load_config
from us_gdp_regime.pipeline import prepare_data

config = load_config(Path("config/default.yaml"))
outputs = prepare_data(config)
outputs

C:\Users\diogo\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


{'series': WindowsPath('data/processed/us_gdp_series.csv'),
 'fred_comparison': WindowsPath('data/models/fred_maddison_growth_comparison.csv'),
 'source_validation_summary': WindowsPath('data/models/source_validation_summary.csv'),
 'source_validation_largest_differences': WindowsPath('data/models/source_validation_largest_differences.csv'),
 'fred_comparison_figure': WindowsPath('figures/fred_maddison_growth_comparison.png')}

## Maddison Extraction

The prepared series constructs total real GDP as real GDP per capita multiplied by population. Constant unit scaling does not affect annual growth rates or log-trend timing.

In [3]:
series = pd.read_csv(outputs["series"])
series.head()

,year,real_gdp,gdp_growth,real_gdp_per_capita,population,source,log_real_gdp
0,1920,1.085155e+09,NaN,10152.927109,106881.0,maddison_2023,20.804989
1,1921,1.054214e+09,-2.851328,9674.880488,108964.0,maddison_2023,20.776061
2,1922,1.105913e+09,4.904101,10009.715277,110484.0,maddison_2023,20.823937
3,1923,1.244264e+09,12.510058,11071.242978,112387.0,maddison_2023,20.941810
4,1924,1.274654e+09,2.442424,11126.712983,114558.0,maddison_2023,20.965941


## Variable Definitions

In [4]:
pd.DataFrame({
    "variable": ["year", "real_gdp", "gdp_growth", "real_gdp_per_capita", "population", "source"],
    "definition": [
        "Calendar year",
        "Maddison-derived real GDP proxy",
        "Annual percent change in real GDP",
        "Maddison real GDP per capita estimate",
        "Maddison population estimate",
        "Source identifier",
    ],
})

,variable,definition
0,year,Calendar year
1,real_gdp,Maddison-derived real GDP proxy
2,gdp_growth,Annual percent change in real GDP
3,real_gdp_per_capita,Maddison real GDP per capita estimate
4,population,Maddison population estimate
5,source,Source identifier


## Missing-Value Checks

In [5]:
series.isna().sum().to_frame("missing_values")

,missing_values
year,0
real_gdp,0
gdp_growth,1
real_gdp_per_capita,0
population,0
source,0
log_real_gdp,0


## Growth-Rate Calculation Checks

In [6]:
check = series[["year", "real_gdp", "gdp_growth"]].copy()
check["computed_growth"] = check["real_gdp"].pct_change() * 100
check["difference"] = check["gdp_growth"] - check["computed_growth"]
check.head(10)

,year,real_gdp,gdp_growth,computed_growth,difference
0,1920,1.085155e+09,NaN,NaN,NaN
1,1921,1.054214e+09,-2.851328,-2.851328,-1.110223e-14
2,1922,1.105913e+09,4.904101,4.904101,0.000000e+00
3,1923,1.244264e+09,12.510058,12.510058,0.000000e+00
4,1924,1.274654e+09,2.442424,2.442424,0.000000e+00
5,1925,1.296540e+09,1.717023,1.717023,0.000000e+00
6,1926,1.372771e+09,5.879562,5.879562,0.000000e+00
7,1927,1.378145e+09,0.391445,0.391445,-5.551115e-17
8,1928,1.385247e+09,0.515378,0.515378,2.220446e-14
9,1929,1.461345e+09,5.493477,5.493477,-2.220446e-14


## FRED/BEA Overlap Comparison

If a local FRED/BEA file is available, the pipeline writes the overlapping growth comparison and diagnostic summary. Missing FRED data does not block the Maddison-based analysis.

In [7]:
comparison_path = config.paths.models_dir / "fred_maddison_growth_comparison.csv"
summary_path = config.paths.models_dir / "source_validation_summary.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison.head())
    if summary_path.exists():
        display(pd.read_csv(summary_path))
else:
    comparison = pd.DataFrame()
    print("FRED/BEA validation file is not available; source validation was skipped.")

,year,growth_maddison,growth_fred,growth_difference,abs_growth_difference
0,1930,-9.492516,-8.507846,-0.984670,0.984670
1,1931,-6.419140,-6.405667,-0.013473,0.013473
2,1932,-15.068670,-12.898624,-2.170046,2.170046
3,1933,-3.395301,-1.236248,-2.159053,2.159053
4,1934,8.370549,10.807915,-2.437366,2.437366


,n_overlap_years,start_year,end_year,growth_correlation,mean_absolute_difference,root_mean_squared_difference
0,93,1930,2022,0.955453,0.626159,1.471334


In [8]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(series["year"], series["gdp_growth"], linewidth=1.2)
ax.set_title("United States real GDP annual growth")
ax.set_xlabel("Year")
ax.set_ylabel("Annual real GDP growth, %")
ax.grid(alpha=0.3)
ax.text(0.01, -0.18, "Source: Maddison Project Database 2023; GDP is per-capita GDP times population.", transform=ax.transAxes, fontsize=9)
fig.tight_layout()

## Export Of Cleaned Data

The cleaned data are written to `data/processed/us_gdp_series.csv`. If FRED/BEA validation is available, overlap diagnostics are written under `data/models/` and the comparison figure under `figures/`.

In [9]:
outputs

{'series': WindowsPath('data/processed/us_gdp_series.csv'),
 'fred_comparison': WindowsPath('data/models/fred_maddison_growth_comparison.csv'),
 'source_validation_summary': WindowsPath('data/models/source_validation_summary.csv'),
 'source_validation_largest_differences': WindowsPath('data/models/source_validation_largest_differences.csv'),
 'fred_comparison_figure': WindowsPath('figures/fred_maddison_growth_comparison.png')}

## Short Conclusion

Maddison is the appropriate default for a 1920 start date because it covers the full historical window. FRED/BEA `GDPCA` is better treated as a validation source from 1929 onward. When the overlap diagnostics are available, use the correlation, absolute differences, and largest-difference years to judge whether the two growth series are close enough for regime analysis.